 TODO Change this description
# Boca Raton Data Cleaning Pipeline
## City-Specific Data Normalization

This notebook handles **Boca Raton specific** data cleaning and normalization:
- Loads raw CSV files from `results_folder/bocaraton/raw_jason/`
- Removes header repetitions and description-only rows
- Standardizes column names and data types
- Applies Boca Raton-specific field mappings
- Outputs clean, normalized data ready for consolidation

**Input**: Raw CSV files from extraction pipeline  
**Output**: Clean, normalized CSV ready for joining with other cities

## Environment Setup

In [1]:
import pandas as pd
from pathlib import Path
import sys

# Use your existing mask_path() if present; otherwise a tiny fallback
if "mask_path" not in globals():
    def mask_path(p: str | Path) -> str:
        p = Path(p)
        parts = p.parts
        # show only the last 3 segments, prefix with "..."
        return str(Path(*(["..."] + list(parts[-3:]))))

# Project paths - need to go up to project root, not stay in cleaning/
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[2]  # Go up 2 levels from cleaning/
else:
    cwd = Path.cwd()
    # If we're in cleaning/, go up 2 levels; if in src/, go up 1 level
    if cwd.name == "cleaning":
        ROOT = cwd.parents[1]
    elif cwd.name == "src":
        ROOT = cwd.parent
    else:
        ROOT = cwd

RESULTS_DIR = ROOT / "results_folder"
BOCA_DIR = RESULTS_DIR / "bocaraton"
CLEAN_DIR = ROOT / "clean_data"
CLEAN_DIR.mkdir(exist_ok=True)

print("ROOT:", mask_path(ROOT))
print("BOCA_DIR:", mask_path(BOCA_DIR))
print("CLEAN_DIR:", mask_path(CLEAN_DIR))
print(f"Looking for JSON files in: {mask_path(BOCA_DIR / 'raw_json')}")

# Verify the path exists
json_dir = BOCA_DIR / "raw_json"
if json_dir.exists():
    print(f"✅ JSON directory exists")
    json_files = list(json_dir.glob("*.json"))
    print(f"Found {len(json_files)} JSON files")
else:
    print(f"❌ JSON directory does not exist: {json_dir}")
    print(f"Available directories in BOCA_DIR:")
    if BOCA_DIR.exists():
        for item in BOCA_DIR.iterdir():
            print(f"  - {item.name}")
    else:
        print(f"  BOCA_DIR itself doesn't exist!")

ROOT: ...\Edilma Projects\LandingAI-Hack\coderisk-sf
BOCA_DIR: ...\coderisk-sf\results_folder\bocaraton
CLEAN_DIR: ...\LandingAI-Hack\coderisk-sf\clean_data
Looking for JSON files in: ...\results_folder\bocaraton\raw_json
✅ JSON directory exists
Found 1 JSON files


## Load Raw Boca Raton Data
Load all CSV files from the tables directory and combine them.

In [2]:
import json
import pandas as pd
from pathlib import Path
from io import StringIO
import re

def extract_tables_from_json_chunks(json_file_path):
    """
    Extract tables from JSON chunks with type='table' and process HTML/markdown content.
    NOW WITH BOCA RATON HEADER FIXES!
    """
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_tables = []
    
    # Look for chunks with type="table"
    if 'chunks' in data:
        for i, chunk in enumerate(data['chunks']):
            if chunk.get('type') == 'table' and 'markdown' in chunk:
                print(f"Processing table chunk {i+1}")
                
                # Extract markdown content
                markdown_content = chunk['markdown']
                
                # Convert markdown table to DataFrame
                df = parse_markdown_table(markdown_content)
                
                if df is not None and not df.empty:
                    # 🔧 APPLY BOCA RATON FIXES
                    df = fix_boca_table_headers(df)
                    df = group_multiline_cases(df)
                    
                    # Add chunk metadata
                    df['chunk_id'] = i
                    df['chunk_type'] = chunk.get('type', 'unknown')
                    all_tables.append(df)
                    print(f"  → Extracted {len(df)} rows after fixes")
                else:
                    print(f"  → No data extracted from chunk {i+1}")
    
    # Concatenate all tables
    if all_tables:
        combined_df = pd.concat(all_tables, ignore_index=True)
        print(f"\n✅ Total extracted: {len(combined_df)} rows from {len(all_tables)} table chunks")
        return combined_df
    else:
        print("❌ No table chunks found")
        return pd.DataFrame()

def parse_markdown_table(markdown_content):
    """
    Parse markdown table content (including HTML tables) into DataFrame.
    Uses more robust parsing for multi-level headers.
    """
    try:
        # First, try to read as HTML table (handles <table> tags)
        if '<table' in markdown_content.lower():
            # Try with header parameter to handle multi-level headers
            try:
                tables = pd.read_html(StringIO(markdown_content), header=[0, 1])
                if tables:
                    df = tables[0]
                    # Flatten multi-level columns
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = [' '.join(col).strip() for col in df.columns.values]
                    return df
            except:
                # Fallback to single header
                tables = pd.read_html(StringIO(markdown_content))
                if tables:
                    return tables[0]
        
        # If not HTML, try parsing as markdown table
        lines = markdown_content.strip().split('\n')
        table_data = []
        headers = None
        
        for line in lines:
            line = line.strip()
            if '|' in line and not line.startswith('|---'):  # Skip header separator
                # Parse table row
                cells = [cell.strip() for cell in line.split('|')]
                # Remove empty first/last cells (common in markdown tables)
                if cells and cells[0] == '':
                    cells = cells[1:]
                if cells and cells[-1] == '':
                    cells = cells[:-1]
                
                if headers is None:
                    headers = cells
                else:
                    table_data.append(cells)
        
        if headers and table_data:
            df = pd.DataFrame(table_data, columns=headers)
            return df
            
    except Exception as e:
        print(f"Error parsing table: {e}")
        return None
    
    return None

In [3]:
def process_boca_table_chunks(json_file_path):
    """
    Boca Raton specific processor implementing ChatGPT playbook rules.
    Handles multi-level headers, case grouping, and line classification.
    """
    print(f"🔧 Processing Boca Raton JSON: {Path(json_file_path).name}")
    
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_processed_rows = []
    
    if 'chunks' in data:
        for i, chunk in enumerate(data['chunks']):
            if chunk.get('type') == 'table' and 'markdown' in chunk:
                print(f"  Processing table chunk {i+1}")
                
                # Parse raw HTML table
                raw_df = parse_html_table(chunk['markdown'])
                if raw_df is not None and not raw_df.empty:
                    # Apply Boca rules
                    processed_rows = apply_boca_rules(raw_df, chunk_id=i, source_file=json_file_path)
                    all_processed_rows.extend(processed_rows)
                    print(f"    → Extracted {len(processed_rows)} processed rows")
    
    if all_processed_rows:
        final_df = pd.DataFrame(all_processed_rows)
        print(f"✅ Total processed: {len(final_df)} rows")
        return final_df
    else:
        print("❌ No data processed")
        return pd.DataFrame()

def parse_html_table(html_content):
    """Simple HTML table parser - gets the raw data matrix"""
    try:
        if '<table' in html_content.lower():
            tables = pd.read_html(StringIO(html_content))
            if tables:
                return tables[0]
    except Exception as e:
        print(f"    ⚠️ Parse error: {e}")
    return None

def apply_boca_rules(raw_df, chunk_id, source_file):
    """
    Apply ChatGPT Boca Raton rules to raw pandas DataFrame.
    Returns list of processed row dictionaries.
    """
    print(f"    🔧 Applying Boca rules to {raw_df.shape} table")
    
    # Rule C: Header consolidation using first two rows
    headers = consolidate_boca_headers(raw_df)
    data_rows = raw_df.iloc[2:].reset_index(drop=True)  # Skip first 2 header rows
    
    # Apply the consolidated headers
    data_rows.columns = headers[:len(data_rows.columns)]
    
    print(f"    🔧 Headers: {headers}")
    print(f"    🔧 Data rows: {len(data_rows)}")
    
    # Rule D: Row classification and processing
    processed_rows = []
    current_case_data = {}
    
    for idx, row in data_rows.iterrows():
        row_dict = classify_and_process_row(row, current_case_data, headers)
        if row_dict:
            # Add metadata
            row_dict['chunk_id'] = chunk_id
            row_dict['source_file'] = str(source_file)
            processed_rows.append(row_dict)
            
            # Update current case context if this is a case header
            if row_dict.get('line_type') == 'case_header':
                current_case_data.update(row_dict)
    
    return processed_rows

def consolidate_boca_headers(df):
    """Rule C: Build single header row from first two rows"""
    if len(df) < 2:
        return list(df.columns)
    
    row1 = df.iloc[0].fillna('').astype(str)
    row2 = df.iloc[1].fillna('').astype(str)
    
    headers = []
    for i in range(len(df.columns)):
        # Prefer row1, use row2 if row1 is empty and row2 has content
        if i < len(row1) and row1.iloc[i].strip():
            headers.append(row1.iloc[i].strip())
        elif i < len(row2) and row2.iloc[i].strip():
            headers.append(row2.iloc[i].strip())
        else:
            headers.append(f"col_{i}")
    
    return headers

def classify_and_process_row(row, current_case_data, headers):
    """Rule D: Classify row type and extract data"""
    row_data = row.fillna('').astype(str)
    
    # Initialize result dictionary
    result = {
        'line_type': 'unknown',
        'violation_id_raw': current_case_data.get('violation_id_raw', ''),
        'violation_type_raw': '',
        'case_status_raw': '',
        'project_raw': '',
        'district_raw': '',
        'address_raw': current_case_data.get('address_raw', ''),
        'parcel_number_raw': current_case_data.get('parcel_number_raw', ''),
        'assigned_to_raw': current_case_data.get('assigned_to_raw', ''),
        'opened_date_raw': current_case_data.get('opened_date_raw', ''),
        'closed_date_raw': current_case_data.get('closed_date_raw', ''),
        'compliance_date_raw': current_case_data.get('compliance_date_raw', ''),
        'resolved_date_raw': current_case_data.get('resolved_date_raw', ''),
        'fee_total_raw': current_case_data.get('fee_total_raw', ''),
        'violation_code_raw': '',
        'violation_text_raw': '',
        'narrative_raw': '',
        'primary_status_raw': current_case_data.get('primary_status_raw', ''),
        'raw_violation_status': ''
    }
    
    # Find Case # column
    case_col_idx = find_column_index(headers, ['Case #', 'Case Number', 'Case'])
    
    # Rule D1: Check if Case # is non-empty -> case_header
    if case_col_idx is not None and case_col_idx < len(row_data):
        case_num = row_data.iloc[case_col_idx].strip()
        if case_num and case_num not in ['', 'nan', 'Case #']:
            result['line_type'] = 'case_header'
            result['violation_id_raw'] = case_num
            
            # Extract case-level fields
            result.update(extract_case_header_fields(row_data, headers))
            result['primary_status_raw'] = result['case_status_raw']
            return result
    
    # Rule D2: Check for violation codes/labels
    violation_col_idx = find_column_index(headers, ['Violation', 'Case Type', 'Code'])
    if violation_col_idx is not None and violation_col_idx < len(row_data):
        violation_text = row_data.iloc[violation_col_idx].strip()
        if is_violation_code_or_label(violation_text):
            result['line_type'] = 'violation_item'
            result['violation_code_raw'], result['violation_text_raw'] = split_violation_field(violation_text)
            
            # Check for violation status
            status_col_idx = find_column_index(headers, ['Violation Status', 'Status'])
            if status_col_idx is not None and status_col_idx < len(row_data):
                result['raw_violation_status'] = row_data.iloc[status_col_idx].strip()
            
            return result
    
    # Rule D3: Check for Description
    for i, cell in enumerate(row_data):
        if cell.strip().startswith('Description:'):
            result['line_type'] = 'narrative'
            result['narrative_raw'] = cell.strip().replace('Description:', '').strip()
            return result
    
    # Skip empty or unclassifiable rows
    if any(cell.strip() for cell in row_data):
        print(f"    ⚠️ Unclassified row: {[cell[:20] for cell in row_data[:3]]}")
    
    return None

def find_column_index(headers, possible_names):
    """Find index of column by multiple possible names"""
    for name in possible_names:
        for i, header in enumerate(headers):
            if name.lower() in header.lower():
                return i
    return None

def extract_case_header_fields(row_data, headers):
    """Extract all case-level fields from a case header row"""
    fields = {}
    
    mapping = {
        'violation_type_raw': ['Case Type', 'Type'],
        'case_status_raw': ['Case Status', 'Status'],
        'project_raw': ['Project'],
        'district_raw': ['District'],
        'address_raw': ['Main Address', 'Address'],
        'parcel_number_raw': ['Parcel'],
        'assigned_to_raw': ['Assigned To', 'Assigned'],
        'opened_date_raw': ['Opened Date', 'Date Opened'],
        'closed_date_raw': ['Closed Date', 'Date Closed'],
        'compliance_date_raw': ['Compliance Date'],
        'resolved_date_raw': ['Resolved Date'],
        'fee_total_raw': ['Violation Fee Total', 'Fee Total', 'Fee']
    }
    
    for field, possible_cols in mapping.items():
        col_idx = find_column_index(headers, possible_cols)
        if col_idx is not None and col_idx < len(row_data):
            fields[field] = row_data.iloc[col_idx].strip()
    
    return fields

def is_violation_code_or_label(text):
    """Rule D2 heuristics: detect violation codes or labels"""
    if not text or text.strip() == '':
        return False
    
    text = text.strip()
    
    # Code patterns
    if any(pattern in text for pattern in ['SEC.', 'IPMC', '§', 'Code']):
        return True
    if re.search(r'\d+(\.\d+)*', text):  # Section numbers
        return True
    if re.search(r'\([A-Z0-9]+\)', text):  # Parenthetical references
        return True
    
    # All caps labels
    if text.isupper() and len(text) > 3:
        return True
    
    return False

def split_violation_field(text):
    """Split violation field into code and description"""
    text = text.strip()
    
    # Look for code patterns at the start
    code_match = re.match(r'^((?:SEC\.|IPMC|§)?\s*\d+(?:\.\d+)*(?:\([A-Z0-9]+\))?)', text)
    if code_match:
        code = code_match.group(1).strip()
        description = text[len(code):].strip()
        return code, description
    
    # If all caps, treat as description
    if text.isupper():
        return '', text
    
    # Default: first word as code, rest as description
    parts = text.split(' ', 1)
    if len(parts) > 1:
        return parts[0], parts[1]
    else:
        return '', text

## Data Cleaning - Remove Headers and Invalid Rows
Clean up Boca Raton specific data issues like repeated headers and description-only rows.

In [4]:
boca_clean = boca_raw.copy()

print("🔍 Before cleaning:")
print(f"Rows: {len(boca_clean)}")
print(f"Null rates (top 5):\n{boca_clean.isna().mean().sort_values(ascending=False).head()}")

# Remove obvious header repeats (rows whose first col literally equals the header name)
if "Main Address" in boca_clean.columns:
    mask_headers = boca_clean["Main Address"].astype(str).str.strip().eq("Main Address")
    rows_before = len(boca_clean)
    boca_clean = boca_clean[~mask_headers]
    print(f"Removed {rows_before - len(boca_clean)} header repeat rows")

# Drop the description-only rows that slipped under "Main Address"
if "Main Address" in boca_clean.columns:
    mask_descr = boca_clean["Main Address"].astype(str).str.startswith("Description:", na=False)
    rows_before = len(boca_clean)
    boca_clean = boca_clean[~mask_descr]
    print(f"Removed {rows_before - len(boca_clean)} description-only rows")

# Convert date columns
date_columns = ["Resolved Date", "Opened Date", "Closed Date", "Compliance Date", "Citation Issued"]
for col in date_columns:
    if col in boca_clean.columns:
        boca_clean[col] = pd.to_datetime(boca_clean[col], errors="coerce")
        print(f"Converted {col} to datetime")

print(f"\n✅ After cleaning: {len(boca_clean)} rows remaining")
print(f"Sample data:\n{boca_clean.head(3)}")

NameError: name 'boca_raw' is not defined

In [5]:
# 🚀 TEST BOCA RATON PROCESSOR
print("🧪 Testing Boca Raton processor with ChatGPT rules...")

# Test on one file first
test_file = json_files[0]
print(f"Testing with: {test_file.name}")

# Process with new Boca rules
boca_df = process_boca_table_chunks(test_file)

if not boca_df.empty:
    print(f"\n📊 Processing Results:")
    print(f"Shape: {boca_df.shape}")
    print(f"Columns: {list(boca_df.columns)}")
    
    # Show line type distribution
    if 'line_type' in boca_df.columns:
        print(f"\n📋 Line Types:")
        print(boca_df['line_type'].value_counts())
    
    # Show sample of each line type
    for line_type in ['case_header', 'violation_item', 'narrative']:
        sample = boca_df[boca_df['line_type'] == line_type]
        if not sample.empty:
            print(f"\n📝 Sample {line_type}:")
            cols_to_show = ['violation_id_raw', 'line_type', 'case_status_raw', 'violation_code_raw', 'violation_text_raw', 'narrative_raw']
            display_cols = [col for col in cols_to_show if col in sample.columns]
            print(sample[display_cols].head(2).to_string())
    
    # Quick validation
    case_headers = boca_df[boca_df['line_type'] == 'case_header']
    print(f"\n✅ Found {len(case_headers)} unique cases")
    print(f"✅ Sample violation IDs: {case_headers['violation_id_raw'].head(5).tolist()}")
    
else:
    print("❌ No data processed")

🧪 Testing Boca Raton processor with ChatGPT rules...
Testing with: boca_JustFOIA_Request_2024-9180-30p.json
🔧 Processing Boca Raton JSON: boca_JustFOIA_Request_2024-9180-30p.json
  Processing table chunk 5
  Processing table chunk 6
    🔧 Applying Boca rules to (4, 11) table
    🔧 Headers: ['CODE-2024-00001', 'Municipal Code Enforcement SEC. 28-1507(2) SEC. 28-1509(A)', 'Closed - Resolved In Violation In Violation', '01/19/2024 01/19/2024', '01/20/2024 01/20/2024', 'R1D', '201 NE 30Th St, Boca Raton, FL 33431', '064347170600602 00 $0.00 $0.00', 'Austin Mann', '01/02/2024', '01/25/2024']
    🔧 Data rows: 2
    ⚠️ Unclassified row: ['CODE-2024-00002', 'Municipal Code Enfor', 'Closed - Resolved In']
    → Extracted 1 processed rows
  Processing table chunk 11
    🔧 Applying Boca rules to (12, 10) table
    🔧 Headers: ['Case #', 'Case Type', 'Case Status', 'Project', 'District', 'Main Address', 'Parcel', 'Assigned To', 'Opened Date', 'Closed Date']
    🔧 Data rows: 10
    → Extracted 10 pr

## 🎯 Let's Check What We Have
Since the test worked, let's see the results and move forward.

In [9]:
# 📊 CHECK RESULTS FROM TEST
print("🔍 Checking what we got from the Boca processor test...")

# Check if boca_df exists and has data
if 'boca_df' in locals() and not boca_df.empty:
    print(f"✅ SUCCESS! We have data:")
    print(f"Shape: {boca_df.shape}")
    print(f"Columns: {list(boca_df.columns)}")
    
    # Show line types
    if 'line_type' in boca_df.columns:
        print(f"\n📋 Line Types Found:")
        print(boca_df['line_type'].value_counts())
        
        # Show sample case headers
        case_headers = boca_df[boca_df['line_type'] == 'case_header']
        if not case_headers.empty:
            print(f"\n✅ Found {len(case_headers)} case headers")
            print("Sample case headers:")
            print(case_headers[['violation_id_raw', 'case_status_raw', 'address_raw']].head(3))
    
    print(f"\n🚀 READY TO PROCESS ALL FILES!")
    
else:
    print("❌ No boca_df found - the test may not have worked")
    print("Available variables:", [v for v in locals().keys() if not v.startswith('_')])

🔍 Checking what we got from the Boca processor test...
✅ SUCCESS! We have data:
Shape: (587, 21)
Columns: ['line_type', 'violation_id_raw', 'violation_type_raw', 'case_status_raw', 'project_raw', 'district_raw', 'address_raw', 'parcel_number_raw', 'assigned_to_raw', 'opened_date_raw', 'closed_date_raw', 'compliance_date_raw', 'resolved_date_raw', 'fee_total_raw', 'violation_code_raw', 'violation_text_raw', 'narrative_raw', 'primary_status_raw', 'raw_violation_status', 'chunk_id', 'source_file']

📋 Line Types Found:
line_type
case_header       247
violation_item    198
narrative         142
Name: count, dtype: int64

✅ Found 247 case headers
Sample case headers:
  violation_id_raw                                    case_status_raw  \
1  CODE-2024-00003                     Closed - Resolved In Violation   
3  CODE-2024-00004  Municipal Code Closed - R1D Enforcement Resolv...   
4  CODE-2024-00005  Closed - Resolved In Violation 01/02/2024 R1D ...   

                                     

In [10]:
# 🚀 PROCESS ALL BOCA FILES & SAVE
print("⚡ Processing ALL Boca Raton files and saving results...")

try:
    # Process all files using our working functions
    all_data = []
    
    print(f"\nProcessing {len(json_files)} JSON files...")
    for i, json_file in enumerate(json_files, 1):
        print(f"\n[{i}/{len(json_files)}] Processing: {json_file.name}")
        df = process_boca_table_chunks(json_file)
        if not df.empty:
            all_data.append(df)
            print(f"  ✅ Got {len(df)} rows")
        else:
            print(f"  ⚠️ No data from this file")
    
    if all_data:
        # Combine all data
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Apply status normalization (if function exists)
        if 'normalize_boca_status' in globals():
            combined_df = normalize_boca_status(combined_df)
        
        print(f"\n🎯 FINAL RESULTS:")
        print(f"Total rows: {len(combined_df)}")
        
        if 'line_type' in combined_df.columns:
            print(f"Line type breakdown:")
            print(combined_df['line_type'].value_counts())
            
            # Count unique cases
            case_headers = combined_df[combined_df['line_type'] == 'case_header']
            print(f"\nUnique cases: {len(case_headers)}")
        
        # Save results
        output_file = CLEAN_DIR / "boca_raton_processed.csv"
        combined_df.to_csv(output_file, index=False)
        print(f"\n💾 Saved detailed data to: {output_file}")
        
        # Create master view (one per case)
        if 'line_type' in combined_df.columns:
            master_df = combined_df[combined_df['line_type'] == 'case_header'].copy()
            master_file = CLEAN_DIR / "boca_raton_master.csv"
            master_df.to_csv(master_file, index=False)
            print(f"💾 Saved master cases to: {master_file}")
            
            print(f"\n✅ SUCCESS! Ready for financial analysis:")
            print(f"  - {len(combined_df)} detailed rows")
            print(f"  - {len(master_df)} unique cases")
        
    else:
        print("❌ No data processed from any files")
        
except Exception as e:
    print(f"❌ Error during processing: {e}")
    import traceback
    traceback.print_exc()

⚡ Processing ALL Boca Raton files and saving results...

Processing 1 JSON files...

[1/1] Processing: boca_JustFOIA_Request_2024-9180-30p.json
🔧 Processing Boca Raton JSON: boca_JustFOIA_Request_2024-9180-30p.json
  Processing table chunk 5
  Processing table chunk 6
    🔧 Applying Boca rules to (4, 11) table
    🔧 Headers: ['CODE-2024-00001', 'Municipal Code Enforcement SEC. 28-1507(2) SEC. 28-1509(A)', 'Closed - Resolved In Violation In Violation', '01/19/2024 01/19/2024', '01/20/2024 01/20/2024', 'R1D', '201 NE 30Th St, Boca Raton, FL 33431', '064347170600602 00 $0.00 $0.00', 'Austin Mann', '01/02/2024', '01/25/2024']
    🔧 Data rows: 2
    ⚠️ Unclassified row: ['CODE-2024-00002', 'Municipal Code Enfor', 'Closed - Resolved In']
    → Extracted 1 processed rows
  Processing table chunk 11
    🔧 Applying Boca rules to (12, 10) table
    🔧 Headers: ['Case #', 'Case Type', 'Case Status', 'Project', 'District', 'Main Address', 'Parcel', 'Assigned To', 'Opened Date', 'Closed Date']
    🔧

In [8]:
# 🔧 QUICK DEBUG - Let's see what's happening
print("🔍 Debugging the Boca processor step by step...")

# Check if we have the right variables
print(f"JSON files available: {len(json_files)}")
if json_files:
    test_file = json_files[0]
    print(f"Testing file: {test_file.name}")
    
    # Check if file exists and can be opened
    try:
        with open(test_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"✅ File loaded successfully")
        print(f"Keys in JSON: {list(data.keys())}")
        
        if 'chunks' in data:
            table_chunks = [chunk for chunk in data['chunks'] if chunk.get('type') == 'table']
            print(f"✅ Found {len(table_chunks)} table chunks")
            
            if table_chunks:
                # Test parsing one chunk
                chunk = table_chunks[0]
                print(f"First chunk keys: {list(chunk.keys())}")
                
                if 'markdown' in chunk:
                    print(f"Markdown length: {len(chunk['markdown'])}")
                    print("First 200 chars of markdown:")
                    print(chunk['markdown'][:200])
                    
                    # Test basic HTML parsing
                    try:
                        if '<table' in chunk['markdown'].lower():
                            from io import StringIO
                            tables = pd.read_html(StringIO(chunk['markdown']))
                            if tables:
                                df = tables[0]
                                print(f"✅ Pandas parsed table: {df.shape}")
                                print(f"Columns: {list(df.columns)}")
                                print("First 2 rows:")
                                print(df.head(2))
                            else:
                                print("❌ No tables found by pandas")
                        else:
                            print("❌ No HTML table tags found")
                    except Exception as e:
                        print(f"❌ Pandas parse error: {e}")
                else:
                    print("❌ No markdown in chunk")
            else:
                print("❌ No table chunks found")
        else:
            print("❌ No chunks in data")
            
    except Exception as e:
        print(f"❌ Error loading file: {e}")
else:
    print("❌ No JSON files found")

🔍 Debugging the Boca processor step by step...
JSON files available: 1
Testing file: boca_JustFOIA_Request_2024-9180-30p.json
✅ File loaded successfully
Keys in JSON: ['markdown', 'chunks', 'splits', 'metadata', 'source_file', 'model']
✅ Found 38 table chunks
First chunk keys: ['id', 'grounding', 'markdown', 'type']
Markdown length: 479
First 200 chars of markdown:
<a id='047dab95-7864-4d9e-b006-24f1cdf1662a'></a>

<table><thead><tr><th>Case #</th><th>Case Type</th><th>Case Status</th><th>Project</th><th>District</th><th>Main Address</th><th>Parcel</th><th>Assig
✅ Pandas parsed table: (0, 10)
Columns: [('Case #', 'Unnamed: 0_level_1'), ('Case Type', 'Violation'), ('Case Status', 'Violation Status'), ('Project', 'Citation Issued'), ('District', 'Compliance Date'), ('Main Address', 'Resolved Date'), ('Parcel', 'Violation Fee Total'), ('Assigned To', 'Unnamed: 7_level_1'), ('Opened Date', 'Unnamed: 8_level_1'), ('Closed Date', 'Unnamed: 9_level_1')]
First 2 rows:
Empty DataFrame
Columns: 

In [ ]:
def normalize_boca_status(df):
    """
    Rule F: Apply status normalization logic
    """
    print("🔧 Applying status normalization...")
    
    df = df.copy()
    df['status_normalized'] = 'Unknown'
    
    for idx, row in df.iterrows():
        primary_status = str(row.get('primary_status_raw', '')).lower()
        violation_status = str(row.get('raw_violation_status', '')).lower()
        closed_date = row.get('closed_date_raw', '')
        resolved_date = row.get('resolved_date_raw', '')
        
        # Rule F1: Closed and Resolved
        if 'closed' in primary_status and 'resolved' in primary_status:
            df.at[idx, 'status_normalized'] = 'Closed/Resolved'
        
        # Rule F2: Active cases
        elif 'in violation' in violation_status and not closed_date:
            df.at[idx, 'status_normalized'] = 'Active'
        
        # Rule F3: Closed cases with dates
        elif resolved_date or closed_date:
            df.at[idx, 'status_normalized'] = 'Closed'
        
        # Rule F4: Default stays Unknown
    
    print(f"Status distribution:")
    print(df['status_normalized'].value_counts())
    return df

def process_all_boca_files():
    """Process all Boca Raton JSON files and combine"""
    print("🚀 Processing ALL Boca Raton files...")
    
    all_data = []
    for json_file in json_files:
        print(f"\nProcessing: {json_file.name}")
        df = process_boca_table_chunks(json_file)
        if not df.empty:
            all_data.append(df)
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Apply status normalization
        combined_df = normalize_boca_status(combined_df)
        
        print(f"\n✅ FINAL RESULTS:")
        print(f"Total rows: {len(combined_df)}")
        print(f"Unique cases: {combined_df[combined_df['line_type'] == 'case_header']['violation_id_raw'].nunique()}")
        print(f"Line type distribution:")
        print(combined_df['line_type'].value_counts())
        
        return combined_df
    else:
        print("❌ No data found in any files")
        return pd.DataFrame()

In [ ]:
# 🎯 PROCESS ALL FILES AND SAVE
print("⚡ QUICK PROCESSING - Let's get this done!")

# Process all files
boca_final = process_all_boca_files()

if not boca_final.empty:
    # Save to clean data folder
    output_file = CLEAN_DIR / "boca_raton_processed.csv"
    boca_final.to_csv(output_file, index=False)
    
    print(f"\n💾 Saved to: {output_file}")
    
    # Create a master records view (one per case)
    case_master = boca_final[boca_final['line_type'] == 'case_header'].copy()
    
    # Add aggregated violation info to master records
    for case_id in case_master['violation_id_raw'].unique():
        if not case_id:
            continue
            
        case_rows = boca_final[boca_final['violation_id_raw'] == case_id]
        violation_items = case_rows[case_rows['line_type'] == 'violation_item']
        narratives = case_rows[case_rows['line_type'] == 'narrative']
        
        # Aggregate violations
        if not violation_items.empty:
            codes = violation_items['violation_code_raw'].dropna()
            texts = violation_items['violation_text_raw'].dropna()
            
            master_idx = case_master[case_master['violation_id_raw'] == case_id].index[0]
            case_master.at[master_idx, 'all_violation_codes'] = '; '.join(codes.unique())
            case_master.at[master_idx, 'all_violation_texts'] = '; '.join(texts.unique())
        
        # Aggregate narratives
        if not narratives.empty:
            all_narratives = narratives['narrative_raw'].dropna()
            master_idx = case_master[case_master['violation_id_raw'] == case_id].index[0]
            case_master.at[master_idx, 'full_narrative'] = '\n'.join(all_narratives.unique())
    
    # Save master view
    master_file = CLEAN_DIR / "boca_raton_master.csv"
    case_master.to_csv(master_file, index=False)
    
    print(f"💾 Master records saved to: {master_file}")
    print(f"\n🎯 DONE! Ready for financial analysis:")
    print(f"  - Detailed data: {len(boca_final)} rows")
    print(f"  - Master cases: {len(case_master)} unique cases")
    print(f"  - Status breakdown: {case_master['status_normalized'].value_counts().to_dict()}")
    
    # Quick financial preview if fee data exists
    if 'fee_total_raw' in case_master.columns:
        fee_data = pd.to_numeric(case_master['fee_total_raw'].str.replace(r'[^\d.]', '', regex=True), errors='coerce')
        valid_fees = fee_data.dropna()
        if not valid_fees.empty:
            print(f"  - Fee data: {len(valid_fees)} cases with fees, total: ${valid_fees.sum():,.2f}")
    
else:
    print("❌ Processing failed")

## Normalize to Standard Schema
Map Boca Raton columns to the standardized schema used across all cities.

In [ ]:
# Standard schema for all cities
normalized_cols = [
    "case_id_raw", "address_raw", "parcel_raw",
    "violation", "violation_status", "citation_issued",
    "compliance_date", "resolved_date", "opened_date", "closed_date",
    "assigned_to", "project", "district", "violation_fee_total",
    "city", "source_file"
]

# Create normalized DataFrame
boca_normalized = pd.DataFrame(index=boca_clean.index, columns=normalized_cols)

# Map Boca Raton specific columns to normalized schema
column_mapping = {
    "Main Address": "address_raw",
    "Resolved Date": "resolved_date", 
    "Opened Date": "opened_date",
    "Closed Date": "closed_date",
    "Compliance Date": "compliance_date",
    "Citation Issued": "citation_issued",
    "Violation": "violation",
    "Violation Status": "violation_status",
    "Parcel": "parcel_raw",
    "Project": "project",
    "Assigned To": "assigned_to",
    "District": "district"
}

# Apply mapping
for boca_col, norm_col in column_mapping.items():
    if boca_col in boca_clean.columns:
        boca_normalized[norm_col] = boca_clean[boca_col]
        print(f"Mapped: {boca_col} -> {norm_col}")

# Add metadata
boca_normalized["city"] = "Boca Raton"
boca_normalized["source_file"] = boca_clean["source_file"]

print(f"\n✅ Normalized schema applied")
print(f"Non-null values per column:")
for col in normalized_cols:
    non_null = boca_normalized[col].notna().sum()
    if non_null > 0:
        print(f"  {col}: {non_null}")

print(f"\nSample normalized data:")
print(boca_normalized[["address_raw", "violation", "resolved_date", "city"]].head(3))

## Save Clean Data
Save the cleaned and normalized Boca Raton data for consolidation.

In [ ]:
# Save cleaned Boca Raton data
output_file = CLEAN_DIR / "boca_raton_clean.csv"
boca_normalized.to_csv(output_file, index=False)

print(f"✅ Saved clean Boca Raton data to: {output_file}")
print(f"📊 Final stats:")
print(f"  - Rows: {len(boca_normalized)}")
print(f"  - Columns: {len(boca_normalized.columns)}")
print(f"  - Date range: {boca_normalized['resolved_date'].min()} to {boca_normalized['resolved_date'].max()}")
print(f"  - Unique addresses: {boca_normalized['address_raw'].nunique()}")

# Quick validation
print(f"\n🔍 Data validation:")
print(f"  - Missing addresses: {boca_normalized['address_raw'].isna().sum()}")
print(f"  - Missing violations: {boca_normalized['violation'].isna().sum()}")
print(f"  - All rows have city='Boca Raton': {(boca_normalized['city'] == 'Boca Raton').all()}")

print(f"\n🎯 Ready for consolidation! Next: run other city cleaning notebooks, then 3_consolidation.ipynb")